<a href="https://colab.research.google.com/github/Ophelie4/pfizer-document-intelligence/blob/main/project-06-advanced-rag-and-open-source-llms/notebooks/Ophelie_Gemini_vs_OpenSource_LLM_Comparison_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task: Evaluate Gemini vs Mistral 7B / Phi-2 / TinyLlama on SDF Doc Queries

**Objective:** hold the RAG retrieval layer completely fixed and swap *only the generator LLM*, so that
every difference we measure is attributable to the model rather than to retrieval.

**Models compared**
| Model | How it runs | Size |
| --- | --- | --- |
| Gemini (`models/gemini-flash-lite-latest`) | Hosted API, needs `GOOGLE_API_KEY` | n/a |
| Mistral-7B-Instruct-v0.2 (Q4_K_M GGUF) | Local, llama.cpp on the Colab GPU | ~4.1 GB |
| Phi-2 (Q4_K_M GGUF) | Local, llama.cpp on the Colab GPU | ~1.8 GB |
| TinyLlama-1.1B-Chat-v1.0 (Q4_K_M GGUF) | Local, llama.cpp on the Colab GPU | ~0.7 GB |

**Experimental design.** One document, one chunking config (SentenceSplitter 200/30), one embedding
model (MiniLM-L6-v2), one retriever (vector, top_k=4). The retrieved chunks are assembled into a single
prompt string, and *that exact same string* is sent to all four models. So retrieval quality is a
constant, not a variable.

**Before running:** Runtime -> Change runtime type -> T4 GPU. Also make sure `GOOGLE_API_KEY` is set in
the Colab Secrets panel (key icon in the left sidebar) with notebook access enabled.

Expect roughly 20-30 minutes end to end, most of it model downloads.

## Step 1: Install packages

In [ ]:
!pip install -q PyMuPDF
!pip install -q llama-index llama-index-embeddings-huggingface
!pip install -q -U google-genai
!pip install -q gdown

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: no GPU. Set Runtime -> Change runtime type -> T4 GPU, then rerun.")

CUDA available: True
GPU: Tesla T4


In [ ]:
# llama-cpp-python with CUDA 12.x prebuilt wheels (same build used in the previous task)
!pip install -q --no-cache-dir llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu123

## Step 2: Load the SDF document

Same source document as every earlier task in this project: the Cytiva AKTA ready flow-kit storage
letter plus two Certificates of Quality (3 pages).

In [ ]:
import os, fitz

PDF_PATH = "/content/sample-sdf-document.pdf"
DRIVE_FILE_ID = "1k_ZRrdSSJLxhMvBnuoaPVYTz05lLmYEN"  # sample-sdf-document (Project 5).pdf

if not os.path.exists(PDF_PATH):
    !gdown "https://drive.google.com/uc?id={DRIVE_FILE_ID}" -O {PDF_PATH} || true

if not os.path.exists(PDF_PATH) or os.path.getsize(PDF_PATH) < 10000:
    print("Automatic download did not work. Run the next cell to upload the PDF by hand.")
else:
    doc = fitz.open(PDF_PATH)
    full_text = "\n".join(page.get_text() for page in doc)
    print(f"Loaded {len(doc)} pages, {len(full_text)} characters")
    print(full_text[:400])

Loaded 3 pages, 4025 characters
Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: ÄKTATM ready Flow Kit Storage Conditions
To Whom It May Concern,
The recommended storage temperature for standard ÄKTA ready flow kits is provided in Section 8.3 of the Operating 
Instructions 28960345 and specified as > +5 C. This recommendation also applies to all modified ÄKTA ready flow kits 
as w


In [ ]:
# FALLBACK ONLY - run this if the gdown cell above failed, then re-run the cell above.
# from google.colab import files
# uploaded = files.upload()   # pick sample-sdf-document.pdf
# import shutil
# shutil.move(list(uploaded.keys())[0], PDF_PATH)

## Step 3: Build the retrieval layer (built once, shared by all four models)

In [ ]:
from llama_index.core import Document, VectorStoreIndex, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import time

Settings.llm = None  # we call each LLM by hand, so LlamaIndex must not try to load a default one
Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

splitter = SentenceSplitter(chunk_size=200, chunk_overlap=30)
nodes = splitter.get_nodes_from_documents([Document(text=full_text)])
print(f"{len(nodes)} chunks")

t0 = time.time()
index = VectorStoreIndex(nodes)
print(f"Index built in {time.time() - t0:.3f}s")

retriever = index.as_retriever(similarity_top_k=4)

LLM is explicitly disabled. Using MockLLM.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

6 chunks
Index built in 0.038s


In [ ]:
QUERIES = [
    "What sterilization method was used for this product?",
    "Summarize the key specifications in this document.",
    "What are the storage conditions for this product?",
]

PROMPT_TEMPLATE = (
    "You are answering questions about a pharmaceutical supplier document.\n\n"
    "CONTEXT:\n{context}\n\n"
    "QUESTION: {question}\n\n"
    "Answer the question using only the context above. If the context does not contain the "
    "answer, say so explicitly. Be specific and include exact values where they appear.\n\n"
    "ANSWER:"
)

# Retrieve ONCE per query. Every model gets a byte-identical prompt.
PROMPTS = {}
RETRIEVAL_LOG = {}

for q in QUERIES:
    t0 = time.time()
    hits = retriever.retrieve(q)
    dt = time.time() - t0
    context = "\n\n".join(f"[chunk {i+1}] {h.node.get_content()}" for i, h in enumerate(hits))
    PROMPTS[q] = PROMPT_TEMPLATE.format(context=context, question=q)
    RETRIEVAL_LOG[q] = {
        "retrieve_seconds": round(dt, 4),
        "n_chunks": len(hits),
        "scores": [round(h.score, 4) for h in hits],
        "top_chunk_excerpt": hits[0].node.get_content()[:220],
        "prompt_chars": len(PROMPTS[q]),
    }
    print(f"\nQ: {q}")
    print(f"   {len(hits)} chunks, scores {RETRIEVAL_LOG[q]['scores']}, "
          f"retrieve {dt:.4f}s, prompt {len(PROMPTS[q])} chars")
    print(f"   top chunk: {RETRIEVAL_LOG[q]['top_chunk_excerpt'][:150]}...")


Q: What sterilization method was used for this product?
   4 chunks, scores [0.3662, 0.3626, 0.2977, 0.1885], retrieve 0.0357s, prompt 3318 chars
   top chunk: Description 
Limits 
Autoclave
Pump tubing
121°C > 15 min 
Gamma Irradiation 
Inlets 
25.0 – 40.0 kGy 
Regulatory Conformance 
Bio-safety
All polymeri...

Q: Summarize the key specifications in this document.
   4 chunks, scores [0.2377, 0.1654, 0.1569, 0.1451], retrieve 0.0128s, prompt 3104 chars
   top chunk: Description
Part Number
Operating Temperature
High Flow Kit F, Modified, ÄKTA ready
29477427
+2 C to +40 C
High Flow Gradient C, Modified, ÄKTA ready
...

Q: What are the storage conditions for this product?
   4 chunks, scores [0.241, 0.1617, 0.1581, 0.1173], retrieve 0.0130s, prompt 3380 chars
   top chunk: Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: ÄKTATM ready Flow Kit Storage Conditions
To Whom It ...


## Step 4: Scoring helpers

`latency`, `tokens/sec` and `finish_reason` are measured directly. Fact recall is a crude keyword
proxy - it checks whether the values that genuinely appear in the source document show up in the
answer. It is a signal, not a grade: read the actual answers before trusting it.

In [ ]:
EXPECTED_FACTS = {
    QUERIES[0]: ["gamma", "irradiat", "kgy", "autoclave", "121"],
    QUERIES[1]: ["flow", "lot", "temperature", "irradiat", "usp"],
    QUERIES[2]: ["+5", "storage", "+2", "+40", "brittle"],
}

RESULTS = []  # one row per (model, query)

def fact_recall(query, answer):
    keys = EXPECTED_FACTS[query]
    low = answer.lower()
    hit = [k for k in keys if k.lower() in low]
    return len(hit), len(keys), hit

def record(model, query, answer, seconds, n_tokens, finish_reason, notes=""):
    hits, total, which = fact_recall(query, answer)
    row = {
        "model": model,
        "query": query,
        "answer": answer.strip(),
        "seconds": round(seconds, 2),
        "output_tokens": n_tokens,
        "tokens_per_sec": round(n_tokens / seconds, 2) if (n_tokens and seconds) else None,
        "answer_chars": len(answer.strip()),
        "finish_reason": finish_reason,
        "fact_recall": f"{hits}/{total}",
        "facts_found": which,
        "notes": notes,
    }
    RESULTS.append(row)
    print(f"\n=== {model} | {query}")
    print(f"    {row['seconds']}s | {n_tokens} tok | {row['tokens_per_sec']} tok/s | "
          f"finish={finish_reason} | facts {row['fact_recall']} {which}")
    print(f"    {row['answer'][:600]}")
    return row

MAX_TOKENS = 300
TEMPERATURE = 0.1

## Step 5: Gemini (hosted API baseline)

In [ ]:
from google.colab import userdata
from google import genai
from google.genai import types

# The google-genai SDK is used directly rather than through LlamaIndex's LLM wrapper.
# The wrapper calls asyncio.run() (conflicts with Colab's running event loop) and raises
# instead of returning text when a response stops at MAX_TOKENS. The SDK returns the
# partial text plus a real finish_reason and real token counts.
client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))

# models/gemini-flash-latest returned repeated 503 "model overloaded" errors, so the lite
# model is used here. Swap it back if flash-latest is responsive when you run this.
GEMINI_MODEL = "models/gemini-flash-lite-latest"

RESULTS = [r for r in RESULTS if not str(r.get("model")).startswith("Gemini")]

for q in QUERIES:
    t0 = time.time()
    resp = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=PROMPTS[q],
        config=types.GenerateContentConfig(
            temperature=TEMPERATURE,
            max_output_tokens=MAX_TOKENS,
        ),
    )
    dt = time.time() - t0
    cand = resp.candidates[0]
    parts = (cand.content.parts if cand.content else None) or []
    text = "".join((p.text or "") for p in parts)
    n_tok = resp.usage_metadata.candidates_token_count
    finish = str(cand.finish_reason)
    record("Gemini Flash Lite", q, text, dt, n_tok, finish,
           notes=f"hosted API {GEMINI_MODEL} via google-genai SDK, network round-trip included")



=== Gemini Flash Lite | What sterilization method was used for this product?
    0.73s | 81 tok | 111.01 tok/s | finish=FinishReason.STOP | facts 5/5 ['gamma', 'irradiat', 'kgy', 'autoclave', '121']
    Based on the provided context, two sterilization methods and their limits are mentioned:

1. **Autoclave** (for Pump tubing): $121^\circ\text{C} > 15\text{ min}$
2. **Gamma Irradiation** (for Inlets): $25.0 \text{--} 40.0\text{ kGy}$

=== Gemini Flash Lite | Summarize the key specifications in this document.
    1.1s | 296 tok | 268.04 tok/s | finish=FinishReason.MAX_TOKENS | facts 4/5 ['flow', 'lot', 'temperature', 'irradiat']
    Based on the provided documents, the key specifications and details include:

*   **High Flow Kit F, Modified, ÄKTA ready**
    *   Part Number: 29477427
    *   Operating Temperature: +2 °C to +40 °C
*   **High Flow Gradient C, Modified, ÄKTA ready**
    *   Part Number: 29184612
    *   Operating Temperature: +2 °C to +40 °C
*   **AKTA ready Low Flow Kit**

## Step 6: The three local GGUF models

Each model is downloaded, loaded with **full GPU offload** (`n_gpu_layers=-1`), run against all three
prompts, then freed before the next one loads. Full offload is a deliberate change from the previous
task, which used `n_gpu_layers=1` and left Mistral running mostly on CPU at ~1.6 tokens/sec.

In [ ]:
import gc
from llama_cpp import Llama

MODELS = [
    {
        "name": "Mistral-7B-Instruct-v0.2 Q4_K_M",
        "path": "/content/mistral.gguf",
        "url": "https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf",
        "mode": "chat",
        "n_ctx": 4096,   # Mistral v0.2 trained on 32k, 4096 is plenty here
    },
    {
        "name": "Phi-2 Q4_K_M",
        "path": "/content/phi2.gguf",
        "url": "https://huggingface.co/TheBloke/phi-2-GGUF/resolve/main/phi-2.Q4_K_M.gguf",
        "mode": "completion",   # Phi-2 is a base model with no chat template
        "n_ctx": 2048,          # Phi-2's native training context
    },
    {
        "name": "TinyLlama-1.1B-Chat-v1.0 Q4_K_M",
        "path": "/content/tinyllama.gguf",
        "url": "https://huggingface.co/TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF/resolve/main/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf",
        "mode": "chat",
        "n_ctx": 2048,          # TinyLlama's native training context
    },
]

for m in MODELS:
    if not os.path.exists(m["path"]):
        print(f"Downloading {m['name']} ...")
        !wget -q --show-progress {m['url']} -O {m['path']}
    print(f"{m['name']}: {os.path.getsize(m['path']) / (1024**3):.2f} GB")

/content/mistral.gg 100%[===================>]   4.07G  88.6MB/s    in 42s     
Mistral-7B-Instruct-v0.2 Q4_K_M: 4.07 GB
/content/phi2.gguf  100%[===================>]   1.67G  79.4MB/s    in 28s     
Phi-2 Q4_K_M: 1.67 GB
/content/tinyllama. 100%[===================>] 637.81M  98.7MB/s    in 7.0s    
TinyLlama-1.1B-Chat-v1.0 Q4_K_M: 0.62 GB


In [ ]:
def run_local_model(spec):
    print(f"\n{'#' * 70}\n# Loading {spec['name']}\n{'#' * 70}")
    t_load = time.time()
    llm = Llama(
        model_path=spec["path"],
        n_gpu_layers=-1,          # full offload to the T4
        n_ctx=spec["n_ctx"],
        verbose=False,
    )
    load_seconds = time.time() - t_load
    print(f"loaded in {load_seconds:.2f}s, context window {spec['n_ctx']} tokens")

    for q in QUERIES:
        prompt = PROMPTS[q]
        t0 = time.time()
        if spec["mode"] == "chat":
            out = llm.create_chat_completion(
                messages=[{"role": "user", "content": prompt}],
                max_tokens=MAX_TOKENS,
                temperature=TEMPERATURE,
            )
            text = out["choices"][0]["message"]["content"]
        else:
            # Phi-2's documented Instruct/Output format
            out = llm(
                f"Instruct: {prompt}\nOutput:",
                max_tokens=MAX_TOKENS,
                temperature=TEMPERATURE,
                stop=["Instruct:", "\n\n\n"],
            )
            text = out["choices"][0]["text"]
        dt = time.time() - t0
        n_tok = out["usage"]["completion_tokens"]
        finish = out["choices"][0].get("finish_reason", "unknown")
        record(spec["name"], q, text, dt, n_tok, finish,
               notes=f"local GGUF, full GPU offload, n_ctx={spec['n_ctx']}, load {load_seconds:.1f}s")

    del llm
    gc.collect()
    torch.cuda.empty_cache()
    print(f"released {spec['name']}")

for spec in MODELS:
    try:
        run_local_model(spec)
    except Exception as e:
        # A failure on one model is recorded and the others still run.
        print(f"{spec['name']} FAILED: {type(e).__name__}: {e}")


######################################################################
# Loading Mistral-7B-Instruct-v0.2 Q4_K_M
######################################################################
loaded in 17.78s, context window 4096 tokens

=== Mistral-7B-Instruct-v0.2 Q4_K_M | What sterilization method was used for this product?
    508.83s | 91 tok | 0.18 tok/s | finish=stop | facts 5/5 ['gamma', 'irradiat', 'kgy', 'autoclave', '121']
    The context indicates that the pump tubing was autoclaved at 121°C for more than 15 minutes and that the inlets were exposed to gamma irradiation with a dose between 25.0 and 40.0 kGy. Therefore, both the pump tubing and the inlets were sterilized using different methods: autoclaving and gamma irradiation, respectively.

=== Mistral-7B-Instruct-v0.2 Q4_K_M | Summarize the key specifications in this document.
    566.83s | 230 tok | 0.41 tok/s | finish=stop | facts 5/5 ['flow', 'lot', 'temperature', 'irradiat', 'usp']
    The document provides information abou

## Step 7: Context-limit test

The three questions above sit comfortably inside every model's window. This cell does the opposite:
it stuffs the **entire document** into the prompt and asks for a long structured summary, so we can
see which models truncate (`finish_reason == "length"`) and which handle it.

In [ ]:
LONG_PROMPT = (
    "You are reading a full pharmaceutical supplier document.\n\n"
    "DOCUMENT:\n" + full_text + "\n\n"
    "Produce a detailed structured summary covering: every product named, every lot and article "
    "number, every date, every test method with its limit and result, the regulatory conformance "
    "statements, and the storage and operating temperatures.\n\nSUMMARY:"
)
print(f"Long prompt: {len(LONG_PROMPT)} chars")

CONTEXT_RESULTS = []
LONG_MAX_TOKENS = 600

# Gemini
t0 = time.time()
resp = client.models.generate_content(
    model=GEMINI_MODEL,
    contents=LONG_PROMPT,
    config=types.GenerateContentConfig(
        temperature=TEMPERATURE,
        max_output_tokens=LONG_MAX_TOKENS,
    ),
)
dt = time.time() - t0
cand = resp.candidates[0]
parts = (cand.content.parts if cand.content else None) or []
text = "".join((p.text or "") for p in parts)
CONTEXT_RESULTS.append({"model": "Gemini Flash Lite", "seconds": round(dt, 2),
                        "context_window": "very large",
                        "output_tokens": resp.usage_metadata.candidates_token_count,
                        "prompt_tokens": resp.usage_metadata.prompt_token_count,
                        "finish_reason": str(cand.finish_reason),
                        "answer_chars": len(text), "answer": text})
print(f"Gemini: {dt:.2f}s, finish={cand.finish_reason}, {len(text)} chars")

# Local models, reloaded one at a time
for spec in MODELS:
    llm = Llama(model_path=spec["path"], n_gpu_layers=-1, n_ctx=spec["n_ctx"], verbose=False)
    t0 = time.time()
    try:
        if spec["mode"] == "chat":
            out = llm.create_chat_completion(
                messages=[{"role": "user", "content": LONG_PROMPT}],
                max_tokens=LONG_MAX_TOKENS, temperature=TEMPERATURE)
            text = out["choices"][0]["message"]["content"]
        else:
            out = llm(f"Instruct: {LONG_PROMPT}\nOutput:", max_tokens=LONG_MAX_TOKENS,
                      temperature=TEMPERATURE, stop=["Instruct:"])
            text = out["choices"][0]["text"]
        dt = time.time() - t0
        CONTEXT_RESULTS.append({
            "model": spec["name"], "seconds": round(dt, 2),
            "context_window": spec["n_ctx"],
            "output_tokens": out["usage"]["completion_tokens"],
            "prompt_tokens": out["usage"]["prompt_tokens"],
            "finish_reason": out["choices"][0].get("finish_reason"),
            "answer_chars": len(text), "answer": text})
        print(f"{spec['name']} (window {spec['n_ctx']}): {dt:.2f}s, "
              f"finish={out['choices'][0].get('finish_reason')}, "
              f"{out['usage']['prompt_tokens']} prompt tok, {len(text)} chars")
    except Exception as e:
        # An overflow here is itself a result: the model's window could not hold the document.
        CONTEXT_RESULTS.append({"model": spec["name"], "context_window": spec["n_ctx"],
                                "error": f"{type(e).__name__}: {e}"})
        print(f"{spec['name']} (window {spec['n_ctx']}) FAILED: {type(e).__name__}: {e}")
    del llm
    gc.collect()
    torch.cuda.empty_cache()

Long prompt: 4343 chars
Gemini: 1.84s, finish=FinishReason.MAX_TOKENS, 1497 chars


KeyboardInterrupt: 

## Step 8: Results summary

In [ ]:
import pandas as pd

df = pd.DataFrame(RESULTS)
summary = (df.groupby("model")
             .agg(avg_seconds=("seconds", "mean"),
                  avg_tokens_per_sec=("tokens_per_sec", "mean"),
                  avg_answer_chars=("answer_chars", "mean"),
                  truncated=("finish_reason", lambda s: sum("length" in str(x).lower() for x in s)))
             .round(2))
recall = df.groupby("model")["fact_recall"].apply(list)
summary["fact_recall_per_query"] = recall
display(summary)
display(df[["model", "query", "seconds", "tokens_per_sec", "fact_recall", "finish_reason"]])

,avg_seconds,avg_tokens_per_sec,avg_answer_chars,truncated,fact_recall_per_query
model,,,,,
Gemini Flash Lite,0.82,173.22,476.00,0,"[5/5, 4/5, 5/5]"
Mistral-7B-Instruct-v0.2 Q4_K_M,524.56,0.26,512.67,0,"[5/5, 5/5, 4/5]"
Phi-2 Q4_K_M,226.01,0.46,535.33,1,"[3/5, 1/5, 1/5]"
TinyLlama-1.1B-Chat-v1.0 Q4_K_M,104.69,2.39,1098.00,1,"[2/5, 2/5, 5/5]"


,model,query,seconds,tokens_per_sec,fact_recall,finish_reason
0,Gemini Flash Lite,What sterilization method was used for this pr...,0.73,111.01,5/5,FinishReason.STOP
1,Gemini Flash Lite,Summarize the key specifications in this docum...,1.10,268.04,4/5,FinishReason.MAX_TOKENS
2,Gemini Flash Lite,What are the storage conditions for this product?,0.64,140.61,5/5,FinishReason.STOP
3,Mistral-7B-Instruct-v0.2 Q4_K_M,What sterilization method was used for this pr...,508.83,0.18,5/5,stop
4,Mistral-7B-Instruct-v0.2 Q4_K_M,Summarize the key specifications in this docum...,566.83,0.41,5/5,stop
5,Mistral-7B-Instruct-v0.2 Q4_K_M,What are the storage conditions for this product?,498.02,0.19,4/5,stop
6,Phi-2 Q4_K_M,What sterilization method was used for this pr...,293.00,1.02,3/5,length
7,Phi-2 Q4_K_M,Summarize the key specifications in this docum...,179.15,0.17,1/5,stop
8,Phi-2 Q4_K_M,What are the storage conditions for this product?,205.87,0.18,1/5,stop
9,TinyLlama-1.1B-Chat-v1.0 Q4_K_M,What sterilization method was used for this pr...,114.38,2.62,2/5,length


In [ ]:
# Full dump - copy everything printed below into the chat to have the write-up built from it.
import json

full_payload = {
    "retrieval": RETRIEVAL_LOG,
    "per_query_results": RESULTS,
    "context_limit_test": CONTEXT_RESULTS,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only",
}
with open("/content/model_comparison_results.json", "w") as f:
    json.dump(full_payload, f, indent=2)

def trim(d, key, n=700):
    d = dict(d)
    if key in d and isinstance(d[key], str) and len(d[key]) > n:
        d[key] = d[key][:n] + f" ...[TRUNCATED, {len(d[key])} chars total]"
    return d

compact = {
    "gpu": full_payload["gpu"],
    "retrieval": RETRIEVAL_LOG,
    "per_query_results": [trim(r, "answer") for r in RESULTS],
    "context_limit_test": [trim(c, "answer", 500) for c in CONTEXT_RESULTS],
}

print("=" * 78)
print("COPY EVERYTHING BELOW THIS LINE")
print("=" * 78)
print(json.dumps(compact, indent=1))
print("=" * 78)
print("END OF RESULTS")

COPY EVERYTHING BELOW THIS LINE
{
 "gpu": "Tesla T4",
 "retrieval": {
  "What sterilization method was used for this product?": {
   "retrieve_seconds": 0.0357,
   "n_chunks": 4,
   "scores": [
    0.3662,
    0.3626,
    0.2977,
    0.1885
   ],
   "top_chunk_excerpt": "Description \nLimits \nAutoclave\nPump tubing\n121\u00b0C > 15 min \nGamma Irradiation \nInlets \n25.0 \u2013 40.0 kGy \nRegulatory Conformance \nBio-safety\nAll polymeric materials in contact with the process fluid comply with United State",
   "prompt_chars": 3318
  },
  "Summarize the key specifications in this document.": {
   "retrieve_seconds": 0.0128,
   "n_chunks": 4,
   "scores": [
    0.2377,
    0.1654,
    0.1569,
    0.1451
   ],
   "top_chunk_excerpt": "Description\nPart Number\nOperating Temperature\nHigh Flow Kit F, Modified, \u00c4KTA ready\n29477427\n+2 C to +40 C\nHigh Flow Gradient C, Modified, \u00c4KTA ready\n29184612\n+2 C to +40 C\nOperating Instructions access link:\nhttps://cdn",
   "prompt_ch

## Observations

_Fill in after running - or paste the JSON dump above into the chat and have it drafted from the real
numbers._

- Speed:
- Accuracy / fact recall:
- Context limit:
- Which model felt most "human":
- Would you trust it with important documents: